# HW3 Part 2




**Course Code:**

**Group number:**

**Student Name:**

**Student ID:**

# Turn Continuity Classification


**Task**: Binary classification — predict whether a spoken turn is **Complete (1)** or **Incomplete (0)**.

**Metric**: Macro-F1 Score.

| Label | Meaning |
|---|---|
| 1 | **Complete** — semantic intent is finished; system can respond |
| 0 | **Incomplete** — intent is unfinished; system should keep listening |


## 0. Environment Setup & Data Loading

In [19]:
!pip install datasets xgboost imbalanced-learn nltk -q

In [20]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Core ML
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix

# Advanced
import xgboost as xgb
from imblearn.over_sampling import SMOTE

# NLP
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)


print("All imports successful!")

All imports successful!


In [21]:
# Load dataset (ensure train.csv and test.csv are in your current directory)
train_path = 'train.csv'
test_path = 'test.csv'

train_df = pd.read_csv(train_path)
public_test_df = pd.read_csv(test_path)

# Basic exploration
print(f"Train size: {len(train_df)}")
print("Label Distribution:\n", train_df['label'].value_counts())
display(train_df.head())

Train size: 1302
Label Distribution:
 label
1    837
0    465
Name: count, dtype: int64


,id,content,label
0,1166,i think we should consider the actually the cl...,1
1,1127,Can you reset my password? My account password...,1
2,1240,im wondering if we need to no wait the test ca...,1
3,1853,"This code needs to be reviewed by tomorrow, do...",0
4,194,"This homework is taking forever, btw the new s...",0


In [23]:
# ── Standardize column names ───────────────────────────────────────────────────

# Based on your files, the text column is named 'content'
# We will normalize it to 'text' for consistency in the pipeline
text_col_mapping = {'content': 'text'}

# Rename 'content' to 'text' if it exists
train_df = train_df.rename(columns=text_col_mapping)
public_test_df = public_test_df.rename(columns=text_col_mapping)

# Ensure 'id' column exists (if not already present)
if 'id' not in train_df.columns:
    train_df['id'] = train_df.index
if 'id' not in public_test_df.columns:
    public_test_df['id'] = public_test_df.index

# ── Label Analysis (Training Set Only) ────────────────────────────────────────

print("Standardization complete.")
print(f"Final columns: {train_df.columns.tolist()}")

if 'label' in train_df.columns:
    counts = train_df['label'].value_counts()
    print("\nLabel distribution (train):")
    print(counts)

    # Calculate imbalance ratio if both classes 0 and 1 exist
    if 0 in counts and 1 in counts:
        ratio = counts[0] / counts[1]
        print(f"\nImbalance ratio (0/1): {ratio:.2f}")
    else:
        print("\nNote: Only one class found in 'label' column.")
else:
    print("\nWarning: 'label' column not found in train_df.")

# Note: public_test_df does not have a 'label' column, skipping its analysis.

Standardization complete.
Final columns: ['id', 'text', 'label']

Label distribution (train):
label
1    837
0    465
Name: count, dtype: int64

Imbalance ratio (0/1): 0.56


## Part 1: Data Balancing

You must implement and compare two methods:
1. **Basic (Required)**: Random Over-sampling.
2. **Advanced (Choose 1+)**: EDA, Back-translation, SMOTE, or Cost-Sensitive Learning.

In [24]:
# -----------------------------------------------------------------
# PHASE 1: Data Balancing
# -----------------------------------------------------------------

def perform_balancing(df, method='random'):
    """
    REQUIRED: method='random' (Random Over-sampling)
    OPTIONAL: method='advanced' (e.g., SMOTE or Cost-Sensitive hooks)
    """
    if method == 'random':
        # ── Step 1: split by class ─────────────────────────────────────────
        #TODO :Complete the function with the parameters
        #choose label 1 or 0
        # df_majority = df[df['label'] == ]
        # df_minority = df[df['label'] == ]
        df_majority = df[df['label'] == 0]
        df_minority = df[df['label'] == 1]

        # ── Step 2: upsample minority to match majority ────────────────────
        #TODO :Complete the function with the parameters
        # df_minority_upsampled = resample( )
        df_minority_upsampled = resample(
            df_minority,
            replace=True,
            n_samples=len(df_majority),
            random_state=42
        )

        # ── Step 3: combine and shuffle ────────────────────────────────────
        #TODO :Complete the function with the parameters
        # df_balanced = df_balanced.sample( )
        df_balanced = pd.concat([df_majority, df_minority_upsampled])
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        return df_balanced

    elif method == 'advanced':
        # [OPTIONAL PHASE 3]: Implement Advanced method
        # Hint: If using SMOTE, remember it must be applied AFTER TfidfVectorizer
        # because SMOTE works on numerical vectors, not raw text.
        return None

# Quick smoke-test
sample_balanced = perform_balancing(train_df, method='random') #choose your method
print("After Random Over-sampling:")
print(sample_balanced['label'].value_counts())

After Random Over-sampling:
label
1    465
0    465
Name: count, dtype: int64


## Part 2: Baseline Classifier (TF-IDF + SVM)

Establish a baseline. Use Macro-F1 as your primary metric.

In [ ]:
# -----------------------------------------------------------------
# PHASE  3: Text Preprocessing (Optional)
# -----------------------------------------------------------------

# ── Original template (kept for reference) ──
# """
# lemmatizer =
# stop_words =
# """
# def preprocess_text(text):
#
#     [OPTIONAL PHASE 3]: Preprocessing Function
#     Hint: Use .str.lower(), or apply a lambda function for NLTK PorterStemmer/WordNetLemmatizer.
#
# """
# # Apply to entire corpus
# train_df['text_clean'] = train_df['text'].apply(preprocess_text)
# public_test_df['text_clean'] = public_test_df['text'].apply(preprocess_text)
#
# print("Sample original :", train_df['text'].iloc[0])
# print("Sample cleaned  :", train_df['text_clean'].iloc[0])
# """

# ── v1 Filled-in implementation (DISABLED) ──
# Reason: this is a turn-completion task. The Complete/Incomplete signal lives in
# the trailing function words ("...where we", "...unless we"), trailing
# punctuation, and truncated last words ("...thi"). Removing stopwords +
# punctuation + lemmatizing destroys exactly those cues (e.g. "No" -> "" because
# "no" is an nltk stopword). Kept here only for the report comparison.
# lemmatizer = WordNetLemmatizer()
# stop_words = set(stopwords.words('english'))
#
#
# def preprocess_text(text):
#     """Lowercase, strip punctuation, drop stopwords, lemmatize."""
#     if not isinstance(text, str):
#         return ""
#     text = text.lower()
#     text = re.sub(r"[^a-z0-9\s]", " ", text)
#     text = re.sub(r"\s+", " ", text).strip()
#     tokens = [
#         lemmatizer.lemmatize(tok)
#         for tok in text.split()
#         if tok not in stop_words and len(tok) > 1
#     ]
#     return " ".join(tokens)


# ── v2 Light implementation (ACTIVE) ──
# Lowercase + whitespace collapse ONLY. Keep stopwords, punctuation, and
# truncated words so the completeness signal survives into TF-IDF / char-ngrams
# / handcrafted end-of-utterance features (see the FeatureBundle cell below).
def preprocess_text(text):
    """Lowercase + collapse whitespace. Punctuation & stopwords preserved."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Apply to entire corpus
train_df['text_clean'] = train_df['text'].apply(preprocess_text)
public_test_df['text_clean'] = public_test_df['text'].apply(preprocess_text)

print("Sample original :", train_df['text'].iloc[0])
print("Sample cleaned  :", train_df['text_clean'].iloc[0])


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2  ─  Model
# ══════════════════════════════════════════════════════════════════════════════
def get_model(model_type='svm'):
    """
    Model Factory for Baseline and Advanced models.
    """
    if model_type == 'svm':
        # REQUIRED PHASE 2 Baseline
        # TODO : complete the function, you could add or change the parameters
        # return SVC(kernel='', probability=True, random_state=42)
        return SVC(kernel='linear', C=1.0, probability=True, random_state=42)

    elif model_type == 'advanced':
        # [OPTIONAL PHASE 3]
        # Hint: Initialize xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss').
        # Try tuning max_depth, n_estimators, and learning_rate.
        # return None
        return xgb.XGBClassifier(
            n_estimators=400,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1,
        )


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2.5  ─  Feature engineering for Turn-Continuity
#  Word TF-IDF (keep stopwords)  +  char_wb n-grams  +  handcrafted end features
#  Drop-in replacement for the bare TfidfVectorizer used below: it exposes
#  .fit_transform() / .transform() so existing cells need only swap the
#  constructor line.
# ══════════════════════════════════════════════════════════════════════════════
import scipy.sparse as sp

# Words that, when they END an utterance, usually mean the thought is unfinished.
FUNCTION_END_WORDS = {
    'the', 'a', 'an', 'to', 'of', 'in', 'on', 'for', 'and', 'but', 'or', 'so',
    'if', 'because', 'unless', 'while', 'when', 'that', 'this', 'these', 'those',
    'my', 'your', 'our', 'their', 'his', 'her', 'its', 'with', 'at', 'as', 'is',
    'are', 'was', 'were', 'be', 'been', 'being', 'we', 'you', 'i', 'he', 'she',
    'they', 'it', 'about', 'into', 'than', 'then', 'where', 'which', 'who',
    'whom', 'whose', 'though', 'although', 'since', 'before', 'after', 'from',
    'by', 'up', 'would', 'should', 'could', 'will', 'can', 'may', 'might', 'do',
    'does', 'did', 'have', 'has', 'had', "don't", "doesn't", "didn't", 'gonna',
    'wanna', 'the', 'some', 'any', 'no',
}
# Short words that ARE complete utterances on their own (don't flag as truncated).
_SHORT_COMPLETE = {'no', 'ok', 'yes', 'hi', 'bye', 'go', 'me', 'us', 'am', 'do',
                    'sup', 'yep', 'nah', 'yup', 'huh', 'wow'}


def extract_handcrafted(texts):
    """Return a sparse matrix of end-of-utterance / shape features per text."""
    rows = []
    for t in texts:
        s = str(t)
        st = s.strip()
        toks = re.findall(r"[a-z0-9']+", s.lower())
        last = toks[-1] if toks else ''
        n = len(toks)
        ends_term = 1.0 if st[-1:] in '.?!' else 0.0
        ends_ellipsis = 1.0 if (st.endswith('...') or st.endswith('…')) else 0.0
        ends_comma = 1.0 if st[-1:] in ',;:-' else 0.0
        end_func = 1.0 if last in FUNCTION_END_WORDS else 0.0
        last_trunc = 1.0 if (last and len(last) <= 3
                             and last not in _SHORT_COMPLETE
                             and last not in FUNCTION_END_WORDS
                             and ends_term == 0.0) else 0.0
        rows.append([
            float(n), float(np.log1p(n)),
            ends_term, ends_ellipsis, ends_comma,
            end_func, last_trunc, float(len(last)),
        ])
    return sp.csr_matrix(np.asarray(rows, dtype=float))


class FeatureBundle:
    """word TF-IDF + char_wb TF-IDF + handcrafted features, hstacked.

    Behaves like a vectorizer: .fit_transform(texts) then .transform(texts).
    """

    def __init__(self):
        self.word_vec = TfidfVectorizer(
            ngram_range=(1, 2),
            stop_words=None,       # keep function words — they carry the signal
            min_df=1,              # keep rare truncated tokens ("thi")
            max_df=0.95,
            sublinear_tf=True,
        )
        self.char_vec = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(2, 5),    # captures truncated word fragments / suffixes
            min_df=2,
            sublinear_tf=True,
        )

    def fit_transform(self, texts):
        texts = list(texts)
        Xw = self.word_vec.fit_transform(texts)
        Xc = self.char_vec.fit_transform(texts)
        Xh = extract_handcrafted(texts)
        return sp.hstack([Xw, Xc, Xh]).tocsr()

    def transform(self, texts):
        texts = list(texts)
        Xw = self.word_vec.transform(texts)
        Xc = self.char_vec.transform(texts)
        Xh = extract_handcrafted(texts)
        return sp.hstack([Xw, Xc, Xh]).tocsr()


def model_scores(model, X):
    """Continuous score for class 1 (decision_function, else predict_proba)."""
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(X)).ravel()
    return np.asarray(model.predict_proba(X))[:, 1]


def best_threshold(y_true, scores, n=200):
    """Threshold on `scores` that maximizes macro-F1 (for imbalanced metric)."""
    y_true = np.asarray(y_true)
    lo, hi = float(np.min(scores)), float(np.max(scores))
    best_t, best_f1 = 0.0, -1.0
    for t in np.linspace(lo, hi, n):
        f1 = f1_score(y_true, (scores >= t).astype(int), average='macro')
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


print("FeatureBundle + threshold helpers ready.")


In [ ]:

# --- Step 1: Internal Validation Setup ---
train_data, val_data = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['label'])


print(f"Train subset : {len(train_data)} samples")
print(f"Val   subset : {len(val_data)} samples")
print("Train label distribution:")
print(train_data['label'].value_counts())

In [ ]:
# TODO: Run your Baseline experiment
# 1. Balance data
# 2. Extract features (TF-IDF)
# 3. Train SVM
# 4. Evaluate using Macro-F1 and Classification Report

print("Running Baseline Experiment...")

# Define MODEL_TYPE locally for the baseline run
# This ensures this cell can be run independently for the baseline, even if kNNUqchf_Fic hasn't been run.
MODEL_TYPE = 'svm'
BALANCE_METHOD = 'random'

# 1. Balance data
balanced_train_data = perform_balancing(train_data.copy(), method=BALANCE_METHOD)
print(f"Balanced train size: {len(balanced_train_data)}  |  label dist:")
print(balanced_train_data['label'].value_counts())

# Pick the text column: prefer the preprocessed one when Phase 3 was run.
TEXT_COL = 'text_clean' if 'text_clean' in balanced_train_data.columns else 'text'

# 2. Extract features (TF-IDF)
# TODO : complete the function, you could add or change the parameters
# vectorizer = TfidfVectorizer(ngram_range=(), stop_words='')
# ── v1 plain TF-IDF (DISABLED) — strips stopwords, drops the completeness cue ──
# vectorizer = TfidfVectorizer(
#     ngram_range=(1, 2),
#     stop_words='english',
#     min_df=2,
#     max_df=0.95,
#     sublinear_tf=True,
# )
# ── v2 ACTIVE: word TF-IDF (keep stopwords) + char n-grams + end features ──
vectorizer = FeatureBundle()
#If applied text preprocessing:
#change balanced_train_data['text'] to balanced_train_data['text_clean']
# X_train = vectorizer.fit_transform(balanced_train_data['text'])
X_train = vectorizer.fit_transform(balanced_train_data[TEXT_COL])
y_train = balanced_train_data['label']

# 3. Train SVM
clf = get_model(MODEL_TYPE)
clf.fit(X_train, y_train)
print(f"Baseline model ({MODEL_TYPE}) trained on {X_train.shape[0]} balanced samples (column: {TEXT_COL}).")

# 4. Evaluate using Macro-F1 and Classification Report
# Prepare validation data
#If applied text preprocessing:
#change balanced_train_data['text'] to balanced_train_data['text_clean']
# X_val = vectorizer.transform(val_data['text'])
X_val = vectorizer.transform(val_data[TEXT_COL])
y_val = val_data['label']

# Make predictions
y_pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')

print(f"=== Baseline: {MODEL_TYPE}| Balancing: {BALANCE_METHOD} ===")
print(classification_report(y_val, y_pred, target_names=['Incomplete(0)', 'Complete(1)']))
print(f"Macro-F1 Score (default 0.5 threshold): {macro_f1:.4f}")

# ── Macro-F1 threshold tuning ──────────────────────────────────────────────
# The competition metric is macro-F1 under class imbalance; the natural 0.5
# decision boundary is rarely optimal. Pick the threshold on the validation
# set (the legitimate model-selection set here) and report the gain.
val_scores = model_scores(clf, X_val)
BASELINE_THRESHOLD, tuned_f1 = best_threshold(y_val, val_scores)
y_pred_tuned = (val_scores >= BASELINE_THRESHOLD).astype(int)
macro_f1_tuned = f1_score(y_val, y_pred_tuned, average='macro')

print(f"\n=== Baseline + tuned threshold ({BASELINE_THRESHOLD:.4f}) ===")
print(classification_report(y_val, y_pred_tuned, target_names=['Incomplete(0)', 'Complete(1)']))
print(f"Macro-F1 (default 0.5)   : {macro_f1:.4f}")
print(f"Macro-F1 (tuned thresh.) : {macro_f1_tuned:.4f}  (+{macro_f1_tuned - macro_f1:.4f})")


## Part 3: Enhancement with K-Fold (Optional)

Improve your results with implement  K-FOLD Cross Validtaion if needed.

In [ ]:
# --- Step 1: Training Pipeline ---

# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 3  ─  Stratified K-Fold Cross-Validation
#               Tests whether the simple split was just a lucky draw.
# ══════════════════════════════════════════════════════════════════════════════

def run_kfold_cv(df, model_type='svm', balance_method='random', n_splits=5, use_custom_preprocessing=False):
    """
    Run Stratified K-Fold CV and return fold scores + mean/std and all y_true/y_pred pairs.

    Parameters
    ----------
    df            : full training DataFrame
    model_type    : type of model to use ('svm' or 'advanced')
    balance_method: method for data balancing ('random' or 'advanced')
    n_splits      : number of folds (default 5)
    use_custom_preprocessing : bool, if True, uses the 'text_clean' column (after preprocess_text),
                                    else uses the original 'text' column.
    """
    text_col = 'text_clean' if (use_custom_preprocessing and 'text_clean' in df.columns) else 'text'

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_scores, all_y_true, all_y_pred = [], [], []
    all_y_score = []  # out-of-fold continuous scores → honest pooled threshold tuning

    print(f"Running {n_splits}-Fold CV  |  model={model_type}  balance={balance_method}  text_col={text_col}")

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df[text_col], df['label']), 1):
        fold_train = df.iloc[train_idx].reset_index(drop=True)
        fold_val   = df.iloc[val_idx].reset_index(drop=True)

        # 1. Balance only the train fold (never the val fold).
        if balance_method == 'random':
            fold_train_bal = perform_balancing(fold_train, method='random')
        else:
            fold_train_bal = fold_train.copy()

        # 2. Fit TF-IDF on the train fold only.
        # ── v1 plain TF-IDF (DISABLED) ──
        # vec = TfidfVectorizer(
        #     ngram_range=(1, 2),
        #     stop_words='english',
        #     min_df=2,
        #     max_df=0.95,
        #     sublinear_tf=True,
        # )
        # ── v2 ACTIVE: word TF-IDF + char n-grams + end features ──
        vec = FeatureBundle()
        X_tr = vec.fit_transform(fold_train_bal[text_col])
        y_tr = fold_train_bal['label'].values

        # 3. (Advanced) SMOTE must run AFTER TF-IDF since it needs numeric vectors.
        if balance_method == 'advanced':
            sm = SMOTE(random_state=42, k_neighbors=5)
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

        X_vl = vec.transform(fold_val[text_col])
        y_vl = fold_val['label'].values

        # 4. Train + predict + score.
        model = get_model(model_type)
        model.fit(X_tr, y_tr)
        y_hat = model.predict(X_vl)

        f1 = f1_score(y_vl, y_hat, average='macro')
        fold_scores.append(f1)
        all_y_true.extend(y_vl)
        all_y_pred.extend(y_hat)
        all_y_score.extend(model_scores(model, X_vl))  # OOF scores
        print(f"  Fold {fold_idx}/{n_splits}  Macro-F1 = {f1:.4f}")

    fold_scores = np.array(fold_scores)
    print(f"\nCV Macro-F1 = {fold_scores.mean():.4f}  ± {fold_scores.std():.4f}")
    print(classification_report(all_y_true, all_y_pred,
                                target_names=['Incomplete(0)', 'Complete(1)']))

    # ── Honest threshold tuning on POOLED out-of-fold scores ──────────────────
    # Every score here was produced by a model that did NOT train on that row,
    # so picking one threshold on the pooled OOF scores is leakage-free.
    all_y_true_arr = np.asarray(all_y_true)
    all_y_score_arr = np.asarray(all_y_score)
    cv_threshold, _ = best_threshold(all_y_true_arr, all_y_score_arr)
    pooled_default_f1 = f1_score(all_y_true_arr, all_y_pred, average='macro')
    pooled_tuned_pred = (all_y_score_arr >= cv_threshold).astype(int)
    pooled_tuned_f1 = f1_score(all_y_true_arr, pooled_tuned_pred, average='macro')
    print(f"Pooled OOF Macro-F1 (default 0.5)   : {pooled_default_f1:.4f}")
    print(f"Pooled OOF Macro-F1 (thresh={cv_threshold:.4f}): {pooled_tuned_f1:.4f}"
          f"  (+{pooled_tuned_f1 - pooled_default_f1:.4f})")

    return {
        'fold_scores': fold_scores,
        'mean': fold_scores.mean(),
        'std': fold_scores.std(),
        'y_true': all_y_true,
        'y_pred': all_y_pred,
        'y_score': all_y_score,
        'cv_threshold': cv_threshold,
        'pooled_tuned_f1': pooled_tuned_f1,
    }


# --- Step 2: Run the two configurations and compare ---
print("\n=== Baseline: SVM + Random Over-sampling ===")
cv_baseline = run_kfold_cv(
    train_df,
    model_type='svm',
    balance_method='random',
    n_splits=5,
    use_custom_preprocessing=('text_clean' in train_df.columns),
)

print("\n=== Advanced: XGBoost + SMOTE ===")
cv_advanced = run_kfold_cv(
    train_df,
    model_type='advanced',
    balance_method='advanced',
    n_splits=5,
    use_custom_preprocessing=('text_clean' in train_df.columns),
)

print("\n── Summary ──")
print(f"Baseline (SVM + RandomOS) : {cv_baseline['mean']:.4f} ± {cv_baseline['std']:.4f}"
      f"  | tuned {cv_baseline['pooled_tuned_f1']:.4f}")
print(f"Advanced (XGB + SMOTE)    : {cv_advanced['mean']:.4f} ± {cv_advanced['std']:.4f}"
      f"  | tuned {cv_advanced['pooled_tuned_f1']:.4f}")


## Part 4: Final Submission

Train on the full dataset using your best found configuration and generate `submission.csv`.

In [ ]:
# TODO: Train final model on the entire training set

# 1. Balance data using the best method found (e.g., 'random')
# Note: This will use the entire train_df, which has 'text_clean' if preprocessing was applied.
final_balanced_train_df = perform_balancing(train_df.copy(), method='random')
print(f"Full balanced training size: {len(final_balanced_train_df)}")
print(final_balanced_train_df['label'].value_counts())

# Pick text column: prefer text_clean when Phase 3 preprocessing ran.
FINAL_TEXT_COL = 'text_clean' if 'text_clean' in final_balanced_train_df.columns else 'text'

# 2. Extract features (TF-IDF) on the full balanced dataset using the preprocessed text
# Use the same TF-IDF parameters (ngram_range, stop_words) as determined during validation/K-Fold
# final_vectorizer = TfidfVectorizer() #TODO: fill in the parameters
# ── v1 plain TF-IDF (DISABLED) ──
# final_vectorizer = TfidfVectorizer(
#     ngram_range=(1, 2),
#     stop_words='english',
#     min_df=2,
#     max_df=0.95,
#     sublinear_tf=True,
# )
# ── v2 ACTIVE: word TF-IDF + char n-grams + end features ──
final_vectorizer = FeatureBundle()
# Ensure to use 'text_clean' column here for consistency with preprocessing if applied
# X_full_train = final_vectorizer.fit_transform(final_balanced_train_df['text'])
X_full_train = final_vectorizer.fit_transform(final_balanced_train_df[FINAL_TEXT_COL])
y_full_train = final_balanced_train_df['label']

# ── Pick FINAL_THRESHOLD on a held-out slice (honest), then refit on ALL data ──
# Carve a stratified holdout from the RAW (pre-balance) train_df so the
# threshold-selection set reflects the real class ratio, not the oversampled one.
_thr_tr, _thr_val = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df['label'])
_thr_tr_bal = perform_balancing(_thr_tr.copy(), method='random')
_thr_vec = FeatureBundle()
_thr_model = get_model('svm')
_thr_model.fit(_thr_vec.fit_transform(_thr_tr_bal[FINAL_TEXT_COL]),
               _thr_tr_bal['label'])
_thr_scores = model_scores(_thr_model, _thr_vec.transform(_thr_val[FINAL_TEXT_COL]))
FINAL_THRESHOLD, _thr_f1 = best_threshold(_thr_val['label'], _thr_scores)
print(f"Selected FINAL_THRESHOLD = {FINAL_THRESHOLD:.4f}  (holdout macro-F1 {_thr_f1:.4f})")

# 3. Train final SVM model
# The model type and hyperparameters should be chosen based on K-Fold results if K-Fold applied
final_model = get_model('svm')
final_model.fit(X_full_train, y_full_train)
print(f"Final model (svm) trained on {X_full_train.shape[0]} samples using column '{FINAL_TEXT_COL}'.")


In [ ]:
# -----------------------------------------------------------------
# PHASE 4: Kaggle Submission
# -----------------------------------------------------------------

def generate_kaggle_submission(model, vectorizer, test_df, output_name='submission.csv',
                               threshold=None):
    """
    Generates the final CSV. Ensure test_df['text'] is processed the same way as training data.
    `threshold` : if given, classify via tuned decision threshold instead of 0.5.
    """
    # Ensure test data is preprocessed before vectorization, just like training data
    # If 'text_clean' exists from previous preprocessing, use it; otherwise, use 'text'
    # X_test = vectorizer.transform(test_df[text])   # original — `text` was an undefined name (NameError)
    text_col = 'text_clean' if 'text_clean' in test_df.columns else 'text'
    X_test = vectorizer.transform(test_df[text_col])
    # ── v1 default 0.5 prediction (DISABLED) ──
    # test_predictions = model.predict(X_test)
    # ── v2 ACTIVE: macro-F1-tuned threshold (falls back to 0.5 if None) ──
    if threshold is None:
        test_predictions = model.predict(X_test)
    else:
        scores = model_scores(model, X_test)
        test_predictions = (scores >= threshold).astype(int)
        print(f"Applied tuned threshold = {threshold:.4f}")

    submission = pd.DataFrame({'id': test_df['id'], 'label': test_predictions})
    submission.to_csv(output_name, index=False)

    # Display first few rows of submission for verification
    print(f"Saved → {output_name}")
    print("Prediction distribution:")
    print(submission['label'].value_counts())
    display(submission.head(10))

    print(f"Success: {output_name} ready for Kaggle upload.")

# Final Step:
generate_kaggle_submission(final_model, final_vectorizer, public_test_df,
                           threshold=FINAL_THRESHOLD)


## Part 5: Advanced Model — Fine-tuned Transformer (RoBERTa)

SPEC §2 allows fine-tuning BERT/RoBERTa as the advanced model. A pretrained
language model directly captures syntactic/semantic *completeness*, which is
exactly this task. Runs on a Colab GPU in a few minutes (Runtime → Change
runtime type → **GPU**); this section regenerates the final `submission.csv`
with the best model.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 5  ─  Advanced Model: Fine-tuned Transformer (DistilRoBERTa)
#  SPEC §2 explicitly allows fine-tuning BERT/RoBERTa as the advanced model.
#  - Uses RAW text (the LM tokenizer needs casing/punctuation/truncation cues).
#  - Class-weighted loss handles imbalance (no row duplication).
#  - Threshold tuned on a held-out split with the same best_threshold() helper.
#  - Regenerates the FINAL submission.csv with this (best) model.
#  Falls back cleanly to CPU (slow) or skips if transformers/torch are missing.
# ══════════════════════════════════════════════════════════════════════════════
import importlib, subprocess, sys

def _ensure(pkgs):
    for p in pkgs:
        mod = p.replace('-', '_')
        try:
            importlib.import_module(mod)
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

_ensure(["transformers", "accelerate", "datasets"])

try:
    import inspect
    import numpy as _np
    import torch
    from torch import nn
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              TrainingArguments, Trainer, DataCollatorWithPadding,
                              set_seed)
    from datasets import Dataset
    _HF_OK = True
except Exception as e:                       # pragma: no cover
    print("Transformers/torch unavailable — skipping transformer section:", repr(e))
    _HF_OK = False

if _HF_OK:
    set_seed(42)
    MODEL_NAME = "distilroberta-base"   # try "roberta-base" / "microsoft/deberta-v3-small" for more
    MAX_LEN, EPOCHS = 96, 4
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {DEVICE} | Model: {MODEL_NAME}")
    if DEVICE == "cpu":
        print("WARNING: no GPU detected — fine-tuning will be slow. "
              "In Colab: Runtime → Change runtime type → GPU.")

    # RAW text (NOT text_clean): the LM tokenizer wants the original signal.
    tr_tx, va_tx, tr_y, va_y = train_test_split(
        train_df["text"].astype(str).tolist(),
        train_df["label"].tolist(),
        test_size=0.2, random_state=42, stratify=train_df["label"])

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)

    def _tok(batch):
        return tok(batch["text"], truncation=True, max_length=MAX_LEN)

    def _make_ds(texts, labels=None):
        d = {"text": list(texts)}
        if labels is not None:
            d["label"] = list(labels)
        ds = Dataset.from_dict(d).map(_tok, batched=True)
        if labels is not None:
            ds = ds.rename_column("label", "labels")
        return ds.remove_columns(["text"])

    ds_tr, ds_va = _make_ds(tr_tx, tr_y), _make_ds(va_tx, va_y)

    # Balanced class weights (cost-sensitive) instead of oversampling.
    cls_w = torch.tensor(len(tr_y) / (2.0 * _np.bincount(tr_y)), dtype=torch.float)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kw):
            labels = inputs.pop("labels")
            out = model(**inputs)
            loss = nn.functional.cross_entropy(
                out.logits, labels, weight=cls_w.to(out.logits.device))
            return (loss, out) if return_outputs else loss

    def _metrics(ep):
        preds = _np.asarray(ep.predictions).argmax(-1)
        return {"macro_f1": f1_score(ep.label_ids, preds, average="macro")}

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    # transformers renamed evaluation_strategy → eval_strategy (~4.41); support both.
    _ta = inspect.signature(TrainingArguments.__init__).parameters
    _eval_kw = "eval_strategy" if "eval_strategy" in _ta else "evaluation_strategy"
    ta_kwargs = dict(
        output_dir="hf_out", per_device_train_batch_size=16,
        per_device_eval_batch_size=32, learning_rate=2e-5,
        num_train_epochs=EPOCHS, save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="macro_f1",
        greater_is_better=True, logging_steps=20, seed=42,
        report_to="none", fp16=(DEVICE == "cuda"))
    ta_kwargs[_eval_kw] = "epoch"
    args = TrainingArguments(**ta_kwargs)

    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
        tokenizer=tok, data_collator=DataCollatorWithPadding(tok),
        compute_metrics=_metrics)
    trainer.train()

    def _prob1(dataset):
        logits = _np.asarray(trainer.predict(dataset).predictions)
        return torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy(), logits

    val_p1, val_logits = _prob1(ds_va)
    HF_THRESHOLD, hf_tuned_f1 = best_threshold(va_y, val_p1)
    hf_default_f1 = f1_score(va_y, val_logits.argmax(-1), average="macro")
    print(f"\n=== Transformer ({MODEL_NAME}) validation ===")
    print(classification_report(va_y, (val_p1 >= HF_THRESHOLD).astype(int),
                                target_names=["Incomplete(0)", "Complete(1)"]))
    print(f"Macro-F1 default(argmax) : {hf_default_f1:.4f}")
    print(f"Macro-F1 tuned(t={HF_THRESHOLD:.3f}) : {hf_tuned_f1:.4f}")

    # Predict test and OVERWRITE the final submission with the best model.
    ds_te = _make_ds(public_test_df["text"].astype(str).tolist())
    te_p1, _ = _prob1(ds_te)
    hf_pred = (te_p1 >= HF_THRESHOLD).astype(int)
    sub = pd.DataFrame({"id": public_test_df["id"], "label": hf_pred})
    sub.to_csv("submission.csv", index=False)
    print(f"\nsubmission.csv OVERWRITTEN by transformer ({MODEL_NAME}).")
    print("Prediction distribution:", sub["label"].value_counts().to_dict())
    display(sub.head(10))


In [ ]:
# Download in Colab
try:
    from google.colab import files
    files.download('submission.csv')
    print("Download started!")
except ImportError:
    print("Not running in Colab — file saved locally as submission.csv")